# x1a_pcarne — Preço Mundial da Carne Bovina (World Bank)

**Grupo no Modelo Econométrico:** `Geral`  
**Definição Operacional:** preço da carne bovina, USD/kg, mundial, mensal*  
**Conceito e Mecanismo Econômico:** Cotação internacional da carne bovina (USD/kg). Representa a atratividade do retorno esperado da atividade pecuária.  

---

### Origem dos Dados Brutos:
- `2_INTI\xdata\chap1\processed\tabular\h1__prices__worldbank__mensal_robustez.csv`

### Observações e Regras Específicas do Documento de Hipóteses:
> - Selecionar price_carne_bovina; anualizar conforme a janela escolhida.
> - Candidato para a dimensão pecuária; não equivale ao preço local do boi gordo.

---
## Protocolo Padronizado de 6 Fases (AED):
1. **Fase 1: Setup & Load** — Importação de pacotes, localização dinâmica da raiz do projeto e inspeção inicial do shape/estrutura.
2. **Fase 2: Data Cleaning & Wrangling** — Padronização de tipos de dados, harmonização para 141 municípios e aplicação estrita das regras da variável.
3. **Fase 3: Estatística Descritiva Básica** — Resumo numérico e distinção econométrica entre zero verdadeiro e ausência de registro.
4. **Fase 4: Análise Univariada** — Histogramas, boxplots de outliers e evolução temporal agregada.
5. **Fase 5: Análise Bivariada** — Padrões espaço-temporais (painel 2001-2024), heterogeneidade por bioma e nota de identificação econométrica.
6. **Fase 6: Feature Engineering** — Código-esqueleto para transformações finais, taxas, defasagens e integração ao painel.


## Fase 1: Setup & Load

Importamos os pacotes essenciais (`pandas`, `numpy`, `matplotlib`, `seaborn`) e localizamos dinamicamente o arquivo CSV bruto a partir da raiz do repositório da tese.

### Dicionário Intuitivo de Colunas:
- **`data`**: Data da cotação mensal.
- **`variavel`**: Identificador ('price_carne_bovina').
- **`valor_original`**: Cotação bruta.
- **`unidade`**: Unidade de medida ($/kg).
- **`preco_nominal`**: Preço nominal em USD/kg.
- **`ln_price`**: Log do preço.
- **`dln_price`**: Variação percentual mensal.


In [ ]:
# =============================================================================
# FASE 1: SETUP & LOAD
# =============================================================================
import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuração visual para gráficos limpos e didáticos
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 10

# Localização dinâmica da raiz do projeto (onde está a pasta '2_INTI')
caminho_atual = Path.cwd().resolve()
raiz_projeto = None
for p in [caminho_atual] + list(caminho_atual.parents):
    if (p / "2_INTI" / "RESEARCH_PROTOCOL.md").is_file() or (p / "AGENTS.md").is_file():
        raiz_projeto = p
        break

if raiz_projeto is None:
    raise FileNotFoundError("Não foi possível localizar a raiz do projeto (onde está 2_INTI/RESEARCH_PROTOCOL.md).")

# Caminho exato do arquivo conforme especificado no documento de hipóteses
caminho_csv = raiz_projeto / r"2_INTI\xdata\chap1\processed\tabular\h1__prices__worldbank__mensal_robustez.csv"

print(f"Raiz do projeto detectada: {raiz_projeto}")
print(f"Carregando dataset de: {caminho_csv}")

if not caminho_csv.exists():
    raise FileNotFoundError(f"Arquivo não encontrado: {caminho_csv}")

# Leitura com suporte a caracteres especiais (BOM / UTF-8)
df = pd.read_csv(caminho_csv, sep=',', encoding='utf-8-sig', low_memory=False)

print(f"Dimensões do dataset (linhas, colunas): {df.shape}")
print("\n--- Primeiras 5 linhas do dataset bruto ---")
df.head()

In [ ]:
print("--- Informações estruturais das colunas e tipos de dados ---")
df.info()

## Fase 2: Data Cleaning & Wrangling

Realizamos a limpeza sistemática dos dados respeitando as regras específicas para `x1a_pcarne`:
- **Atenção:** Selecionar price_carne_bovina; anualizar conforme a janela escolhida.
- **Atenção:** Candidato para a dimensão pecuária; não equivale ao preço local do boi gordo.


In [ ]:
# =============================================================================
# FASE 2: DATA CLEANING & WRANGLING
# =============================================================================
df.columns = df.columns.str.strip().str.lower().str.replace('\ufeff', '', regex=False)
df['data'] = pd.to_datetime(df['data'], errors='coerce')
df['ano'] = df['data'].dt.year

df = df[df['variavel'] == 'price_carne_bovina'].copy()
df['preco_nominal'] = pd.to_numeric(df['preco_nominal'], errors='coerce')

# REGRA DO TXT: 'Selecionar price_carne_bovina; anualizar conforme a janela escolhida.'
# 'Candidato para a dimensão pecuária; não equivale ao preço local do boi gordo.'
print(f"Registros de carne bovina: {len(df)}")

## Fase 3: Estatística Descritiva Básica

Calculamos os momentos da distribuição (média, desvio, mediana, percentis) e abordamos a questão econométrica crucial:
**A ausência de linha/valor representa zero verdadeiro ou falta de registro?**


In [ ]:
# =============================================================================
# FASE 3: ESTATÍSTICA DESCRITIVA BÁSICA
# =============================================================================
print("Descritiva da carne bovina (USD/kg):")
print(df['preco_nominal'].describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95]))

## Fase 4: Análise Univariada

Examinamos o formato da distribuição (assimetria, caudas longas), a presença de valores extremos (outliers) e as tendências agregadas da variável ao longo do tempo.


In [ ]:
# =============================================================================
# FASE 4: ANÁLISE UNIVARIADA
# =============================================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(df['preco_nominal'].dropna(), bins=30, kde=True, ax=axes[0], color='crimson')
axes[0].set_title("Distribuição do Preço da Carne (USD/kg)")

sns.lineplot(data=df, x='data', y='preco_nominal', ax=axes[1], color='darkred')
axes[1].set_title("Série Histórica Mensal (USD/kg)")
plt.tight_layout()
plt.show()

## Fase 5: Análise Bivariada (Espaço-Temporal & Biomas)

Exploramos a dimensão longitudinal (painel de municípios de Mato Grosso entre 2001 e 2024) e cruzamentos geográficos com biomas (Amazônia, Cerrado e Pantanal).

> **Nota Econométrica sobre Efeitos Fixos:**  
> Características puramente temporais (como preços mundiais) são absorvidas por efeitos fixos de ano ($\lambda_t$).  
> Características puramente espaciais invariantes no tempo (como biomas, relevo e aptidão GAEZ) são absorvidas por efeitos fixos municipais ($\alpha_i$).  
> Para estimar seus efeitos, exploramos interações entre as dimensões tempo e espaço ($Preço_t \times Exposição_i$).


In [ ]:
# =============================================================================
# FASE 5: ANÁLISE BIVARIADA
# =============================================================================
df_painel = df[(df['ano'] >= 2001) & (df['ano'] <= 2024)]
plt.figure(figsize=(10, 4))
sns.lineplot(data=df_painel, x='data', y='preco_nominal', color='brown')
plt.title("Preço Internacional da Carne Bovina no Período do Painel (2001-2024)")
plt.ylabel("USD/kg")
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

## Fase 6: Feature Engineering — Preço da Carne e Termos de Troca Pecuária/Grãos

### 1. Step 1: Harmonização Temporal e Janela de Decisão Pecuária
- **Periodicidade Original:** Mensal (World Bank, USD/kg de carne bovina).
- **Janela de Decisão Pecuária:** Período de Maio a Outubro do ano $t-1$ (estação seca em MT). É nessa época que ocorrem os maiores abates, descarte de fêmeas e planejamento de reforma/expansão de pastagens para o início das chuvas.

### 2. Step 2: Derivações e Relação de Troca
- **Defasagem $t-1$ e Logaritmo:** $\ln(P_{carne, t-1})$ captura o retorno esperado que orienta a compra de bezerros e a retenção de matrizes.
- **Termos de Troca Carne vs Soja:** Mede o poder de atração da terra entre pecuária e agricultura de grãos. Se a soja sobe mais que a carne, a pressão para converter pasto em lavoura aumenta.
- **Interação Espacial Shift-Share:** Como o preço internacional é invariante no espaço, multiplicamos $\ln(P_{carne, t-1})$ pela aptidão de pastagens do município para gerar variação longitudinal em painel com efeito fixo de ano.


In [ ]:
# =============================================================================
# FASE 6: FEATURE ENGINEERING (TRATAMENTOS FINAIS E HARMONIZAÇÃO)
# =============================================================================
# 1. Anualização da Série de Carne Bovina (USD/kg)
pcarne_anual = df.groupby('ano').agg(
    pcarne_media_anual=('preco_nominal', 'mean')
).reset_index()

# 2. Janela de Decisão Pecuária (Maio a Outubro = meses 5 a 10)
pcarne_janela = df[df['data'].dt.month.isin([5, 6, 7, 8, 9, 10])].groupby('ano').agg(
    pcarne_janela_seca=('preco_nominal', 'mean')
).reset_index()

pcarne_harmonizado = pd.merge(pcarne_anual, pcarne_janela, on='ano', how='left')

# 3. Defasagem Temporal de 1 ano (t-1)
pcarne_harmonizado['pcarne_lag1'] = pcarne_harmonizado['pcarne_janela_seca'].shift(1)
pcarne_harmonizado['ln_pcarne_lag1'] = np.log(pcarne_harmonizado['pcarne_lag1'])

# 4. Termos de Troca Relativos (Simulação de Interação Shift-Share com Aptidão Pasto)
# choque_carne_it = ln_pcarne_lag1 * aptidao_pastagem_i
pcarne_harmonizado['choque_shift_share_carne'] = pcarne_harmonizado['ln_pcarne_lag1'] * 0.70

painel_carne = pcarne_harmonizado[(pcarne_harmonizado['ano'] >= 2001) & (pcarne_harmonizado['ano'] <= 2024)].copy()

print("Preços Mundiais de Carne Bovina Anualizados e Defasados:")
print(painel_carne[['ano', 'pcarne_media_anual', 'pcarne_lag1', 'ln_pcarne_lag1']].tail(8))